## 10.4 LSTM - Pytorch 实现

#### 1、本节的核心目标

##### 这一节我们要解决 3 个非常重要的问题

- PyTorch 为什么把四个门合并计算？
- `nn.LSTM` 的 weight 为什么是 4 倍？
- `output / h_n / c_n` 到底是什么？

#### 2、PyTorch 中 LSTM 的基本使用

##### 2.1 最基础的使用方式

In [1]:
import torch 
import torch.nn as nn

lstm = nn.LSTM(
    input_size = 32, # 输入特征的维度
    hidden_size= 64, # 隐藏状态的维度
    batch_first=True, # 输入和输出的第一维是batch_size
)

X = torch.randn(100, 10, 32) # (batch_size, seq_length, input_size)
output, (h_n, c_n) = lstm(X)

##### 2.2 输出结果解释（必须掌握）
**（1）output**

👉 表示：

>每一个时间步的隐藏状态 $h_t$

In [5]:
output.shape # (100, 10, 64) 最后一层中每个时间步的输出的汇总

torch.Size([100, 10, 64])

**（2）h_n**

👉 表示：

> 最后一个时间步的隐藏状态

In [6]:
h_n.shape # (num_layers, 100, 64) 每一层最后一个时间步的隐藏状态的汇总

torch.Size([1, 100, 64])

**（3）c_n**

👉 表示：

> 最后一个时间步的细胞状态

In [7]:
c_n.shape # (num_layers, 100, 64) 每一层最后一个时间步的细胞状态的汇总

torch.Size([1, 100, 64])

##### 2.3 为什么 h_n 多了一维？

这个维度其实是：

```python
(num_layers × num_directions)
```

例如：

- 单层 LSTM → `1`
- 双向 LSTM → `2`
- 2 层双向 → `4`

所以：

```python
h_n.shape = (layers × directions, batch, hidden_size)
```

#### 3、最关键点：四个门为什么合并计算 🔥

##### 3.1 理论 vs 实现

我们在理论中是这样计算的：

- Forget Gate
- Input Gate
- Candidate
- Output Gate

👉 4 次矩阵乘法

但 PyTorch 并不是这样做的。


##### 3.2 PyTorch 的做法（核心优化）

PyTorch 会把四个门一次性算出来：

$Z = W_{ih}x_t + W_{hh}h_{t-1} + b$

其中：

$Z \in \mathbb{R}^{4d_h}$


##### 3.3 然后切分成四部分

```python
f_t, i_t, g_t, o_t = torch.chunk(Z, 4, dim=1)
```


##### 3.4 再分别激活

```python
f_t = sigmoid(f_t)
i_t = sigmoid(i_t)
o_t = sigmoid(o_t)
g_t = tanh(g_t)
```


##### 3.5 为什么这样更好？

👉 原因非常重要：

- 减少 4 次矩阵乘法 → 只做 1 次
- GPU 并行效率更高
- 内存访问更连续
- 更符合深度学习框架优化方式


#### 4、权重矩阵为什么是 4 倍

##### 4.1 权重 shape

在 PyTorch 中：

```python
lstm.weight_ih_l0.shape = (4 * hidden_size, input_size)
lstm.weight_hh_l0.shape = (4 * hidden_size, hidden_size)
```


##### 4.2 举个例子

假设：

- `hidden_size = 5`
- `input_size = 4`

那么：

```python
weight_ih = (20, 4)
weight_hh = (20, 5)
```


##### 4.3 为什么是 4 倍？

因为四个门的计算合并了。

因为：

👉 一个矩阵同时计算：

- Forget Gate
- Input Gate
- Candidate
- Output Gate


##### 4.4 对应关系

切分后：

```python
前 5 行   -> Forget Gate
中间 5 行 -> Input Gate
中间 5 行 -> Candidate
后 5 行   -> Output Gate
```

#### 5、多层 LSTM

##### 5.1 定义

```python
lstm = nn.LSTM(input_size=4, hidden_size=5, num_layers=2)
```


##### 5.2 核心理解

**（1）本质一：不是“更长的时间”，而是“更深的网络”**

多层 LSTM = 在“时间维度”之外，再增加“深度维度”

- 单层 LSTM：单个时间步简单计算，只在时间维传播
- 多层 LSTM：在每个时间步内部，还会向上层传递，然后再在时间维传播

👉 可以理解为：

深度维（纵向） + 时间维（横向）


**（2）本质二：每一层都在“重新编码序列”**

- 第一层：处理原始输入 $x_t$
- 第二层：处理“已经被编码过的序列”

👉 所以：

越往上的层，特征越抽象、语义越强

类似 CNN：

- 第一层：边缘
- 高层：语义


**（3）本质三：每一层都有“自己的记忆系统”**

每一层都有独立的：

- $h_t$
- $C_t$

👉 所以不是共享的，而是：

```python
Layer 1: (h, C)
Layer 2: (h, C)
Layer 3: (h, C)
```


**（4）本质四：数据流动方式（最重要🔥）**

在某一个时间步 $t$：

```python
x_t
 ↓
Layer 1 → h_t^(1)
 ↓
Layer 2 → h_t^(2)
 ↓
Layer 3 → h_t^(3)   ← 最终 output
```

👉 然后：

时间继续 → $t+1$

🧠 总结：

多层 LSTM = 时间维递归 + 层间堆叠，每一层都在重新理解序列


##### 5.3 shape 变化

**（1）output 的维度**

多层 LSTM：

`output` 的维度是不变的（和单层一样）

👉 不会因为层数增加而改变

👉 PyTorch 的设计是：

只返回“最后一层”的所有时间步输出

而不是：

- 所有层拼接 ❌
- 所有层相加 ❌


**（2）h_n / c_n 的维度**

多层 LSTM：

```python
h_n.shape = (num_layers, B, d_h)
c_n.shape = (num_layers, B, d_h)
```

$h_t$ 和 $C_t$ 存的是最后一个时间步的状态。

所以会包含每一层的最后一个时间步。

#### 6、双向 LSTM（Bidirectional）

##### 6.1 定义

```python
lstm = nn.LSTM(input_size=4, hidden_size=5, bidirectional=True)
```


##### 6.2 发生了什么

👉 两个方向：

- 正向 → 从左到右
- 反向 → 从右到左


##### 6.3 核心理解

**（1）本质一：同时看“过去”和“未来”**

普通 LSTM：

```python
x1 → x2 → x3 → x4
```

双向 LSTM：

```python
正向： x1 → x2 → x3 → x4
反向： x1 ← x2 ← x3 ← x4
```

👉 每个时间步都有两个信息来源：

- 左边（过去）
- 右边（未来）


**（2）本质二：两个独立的 LSTM**

双向 LSTM = 两个完全独立的 LSTM：

- Forward LSTM（正向）
- Backward LSTM（反向）

它们：

- 参数不同 ❗
- 状态不同 ❗
- 完全独立训练 ❗


**（3）本质三：输出是拼接（不是相加）**

在时间步 $t$：

$h_t = [h_t^{\rightarrow}; h_t^{\leftarrow}]$

👉 所以：

```python
output_dim = 2 * d_h
```


**（4）本质四：为什么更强？**

因为很多任务是“上下文相关”的：

例如：

```python
I saw a bank
```

`bank` 是“河岸”还是“银行”？

👉 要继续看后面的内容！


**（5）本质五：什么时候用？**

适用于：

- NLP（文本分类 / NER / 翻译）
- 语音识别
- 上下文强依赖任务

不适用于：

- 实时预测（因为看不到未来）


##### 6.4 维度变化

**（1）output 的维度**

```python
output.shape = (B, T, 2 * d_h)
```

✅ 为什么是 2 倍？

因为每个时间步都有两个方向的输出：

- forward: $h_t^{\rightarrow}$
- backward: $h_t^{\leftarrow}$

然后拼接：

$h_t = [h_t^{\rightarrow}; h_t^{\leftarrow}]$


**（2）h_n / c_n 的维度**

```python
h_n.shape = (2, B, d_h)
c_n.shape = (2, B, d_h)
```

✅ 为什么是 2？

因为有两个方向：

- `0` → forward
- `1` → backward

⚠️ 重点（非常容易错🔥）

- `forward_last` → 最后时间步 $h_T / c_T$
- `backward_last` → 第一个时间步 $h_1 / c_1$

👉 因为反向是从右往左计算的！

#### 7、多层 + 双向 LSTM

##### 7.1 定义

```python
nn.LSTM(
    input_size=4,
    hidden_size=5,
    num_layers=3,
    bidirectional=True
)
```


##### 7.2 维度变化

**（1）output 的维度**

```python
output.shape = (B, T, 2 * d_h)
```

👉 仍然只和方向有关，不和层数有关


**（2）h_n / c_n 的维度**

```python
h_n.shape = (3 * 2, B, d_h)
c_n.shape = (3 * 2, B, d_h)
```